[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S10/2_Practica/en_clase/S10_01_red_desde_cero_en_clase.ipynb)

# S10 · Práctica 1 — Una red neuronal desde cero, solo con NumPy · versión en clase

**Tiempo estimado:** 40 min en clase · **Sprint 10:** Redes neuronales bien entrenadas (UD 4.1)

📖 *¿El código se hace largo? Léelo [explicado por escrito](https://github.com/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S10/2_Practica/codigo_explicado/S10_01_red_desde_cero.md): recorrido, glosario y qué hace cada celda.* <!--codigo-explicado-->

**Contexto TurisData.** TurisData Canarias quiere saber si una red neuronal "vale la pena"
frente a los modelos clásicos que ya conoces. Antes de usar Keras, vamos a abrir la caja:
construirás una red diminuta con tus propias manos para entender **qué hace** por dentro
(así, cuando algo falle en Keras, sabrás por dónde mirar).

**Al terminar sabrás:**
1. Qué es una neurona artificial y para qué sirve la función de activación.
2. Cómo una red calcula una predicción (propagación hacia delante).
3. Cómo aprende: error, gradiente y propagación hacia atrás (*backpropagation*), sin fórmulas de memoria.
4. Entrenar una red pequeña sobre un problema que un modelo lineal no puede resolver.

> 🏫 **Versión «en clase»** · ⏱ ≈ 40 min <!--reparto-->
>
> - Es **la parte de este cuaderno que se hace en el aula**, con el docente: ejercicios **1, 2, 3**.
> - Lo demás (ejercicios 🏠 y mini-reto) está en el [cuaderno completo](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S10/2_Practica/para_casa/S10_01_red_desde_cero.ipynb), para hacer en casa.


In [ ]:
# Preparación (ejecuta esta celda siempre la primera)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos", "../../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_circles
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

SEMILLA = 42
np.random.seed(SEMILLA)
plt.rcParams["figure.figsize"] = (6, 4)

## 1. La neurona: una suma ponderada y un "interruptor suave"

**Idea.** Una neurona artificial hace tres cosas:
1. Multiplica cada entrada por un **peso** (cuánto le importa esa entrada).
2. Suma todo y añade un **sesgo** (*bias*, un empujón inicial).
3. Pasa el resultado por una **función de activación** (un "interruptor suave" que decide cuánta señal deja pasar).

Ejemplo de juguete: una neurona que estima la "probabilidad de cancelar" a partir de
la antelación (en cientos de días) y de si la tarifa es flexible (1) o no (0).

In [ ]:
def sigmoide(z):
    """Convierte cualquier número en un valor entre 0 y 1 (sirve como probabilidad)."""
    return 1 / (1 + np.exp(-z))

x = np.array([1.5, 1.0])        # antelación 150 días, tarifa flexible
pesos = np.array([0.8, 1.2])    # inventados: la tarifa flexible pesa más
sesgo = -1.0

z = np.dot(x, pesos) + sesgo    # suma ponderada
print("z =", z, "→ salida de la neurona =", round(float(sigmoide(z)), 3))

**Ejercicio 1 · 🏫 en clase.** Escribe la función `neurona(x, pesos, sesgo)` que devuelva la salida
de la neurona (suma ponderada + sesgo, pasada por `sigmoide`).

In [ ]:
# TODO: Usa np.dot(x, pesos) + sesgo y aplica sigmoide al resultado
...

salida_ejemplo = neurona(np.array([1.5, 1.0]), np.array([0.8, 1.2]), -1.0)
assert abs(salida_ejemplo - 0.8022) < 1e-3, "Revisa neurona(): debe ser sigmoide(x·pesos + sesgo). Esperado ≈ 0.802"
assert neurona(np.array([0.0, 0.0]), np.array([5.0, 5.0]), 0.0) == 0.5, "Con entradas 0 y sesgo 0 la sigmoide da 0.5"

## 2. Funciones de activación

**Idea.** Sin activación, apilar capas no sirve de nada: una suma de sumas ponderadas sigue
siendo una suma ponderada (un modelo lineal). La activación introduce **curvas**, y con curvas
la red puede dibujar fronteras complicadas.

- **Sigmoide:** salida entre 0 y 1. Típica en la capa final de una clasificación binaria.
- **ReLU:** deja pasar lo positivo y anula lo negativo. Es la habitual en capas ocultas.
- **Tanh:** como la sigmoide pero entre −1 y 1.

In [ ]:
def tanh(z):
    return np.tanh(z)

z = np.linspace(-6, 6, 200)
plt.plot(z, sigmoide(z), label="sigmoide")
plt.plot(z, tanh(z), label="tanh")
plt.plot(z, np.maximum(0, z), label="ReLU")
plt.ylim(-1.5, 3); plt.axhline(0, color="gray", lw=0.5); plt.axvline(0, color="gray", lw=0.5)
plt.xlabel("valor de entrada z"); plt.ylabel("salida de la activación")
plt.title("Tres funciones de activación"); plt.legend(); plt.show()

**Ejercicio 2 · 🏫 en clase.** Define `relu(z)` (devuelve `z` si es positivo y 0 si no) y `relu_derivada(z)`
(vale 1 donde `z > 0` y 0 en el resto). La derivada nos dirá más adelante "por dónde
fluye el aprendizaje": una neurona ReLU apagada (z ≤ 0) no aprende en esa muestra.

In [ ]:
# TODO: np.maximum(0, z) para relu; (z > 0) convertido a float para la derivada
...

_prueba = np.array([-2.0, 0.0, 3.0])
assert np.array_equal(relu(_prueba), [0.0, 0.0, 3.0]), "relu debe anular los negativos y dejar pasar los positivos"
assert np.array_equal(relu_derivada(_prueba), [0.0, 0.0, 1.0]), "relu_derivada vale 1 solo si z > 0"

## 3. Un problema que un modelo lineal no resuelve

Generamos dos "anillos" de puntos (uno dentro del otro). Piensa en ellos como clientes de
dos tipos con un comportamiento que no se separa con una línea recta. Una regresión logística
(modelo lineal) se quedará en el azar.

In [ ]:
X, y = make_circles(n_samples=600, noise=0.10, factor=0.45, random_state=SEMILLA)
X_ent, X_val, y_ent, y_val = train_test_split(X, y, test_size=0.3, random_state=SEMILLA, stratify=y)

lineal = LogisticRegression().fit(X_ent, y_ent)
print("Exactitud de un modelo lineal en validación:", round(lineal.score(X_val, y_val), 3))

plt.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", s=12)
plt.title("Dos grupos que ninguna recta separa"); plt.xlabel("variable 1"); plt.ylabel("variable 2"); plt.show()

## 4. Propagación hacia delante: de las entradas a la predicción

Nuestra red tendrá: **2 entradas → capa oculta (ReLU) → 1 salida (sigmoide)**.
Una capa es simplemente **muchas neuronas a la vez**, y se calcula con una multiplicación de matrices:

`Z1 = X · W1 + b1` → `A1 = relu(Z1)` → `Z2 = A1 · W2 + b2` → `A2 = sigmoide(Z2)`

`A2` es la probabilidad predicha de pertenecer al grupo 1.

**Propagación hacia atrás.** Para aprender medimos el error (pérdida) y preguntamos:
*"si muevo un poquito este peso, ¿el error sube o baja, y cuánto?"*. Esa respuesta es el **gradiente**.
La regla de la cadena permite calcularlo capa por capa **desde la salida hacia la entrada**
(de ahí "hacia atrás"). Después movemos cada peso un pasito en dirección contraria al gradiente:
es el **descenso de gradiente**.

In [ ]:
def entropia_cruzada(y_real, prob, eps=1e-9):
    """Pérdida media de una clasificación binaria (menor = mejor)."""
    prob = np.clip(prob, eps, 1 - eps)
    return -np.mean(y_real * np.log(prob) + (1 - y_real) * np.log(1 - prob))

print("Si predices siempre 0.5, la pérdida es", round(entropia_cruzada(np.array([0, 1, 1, 0]), np.full(4, 0.5)), 3), "(= ln 2)")
print("Si aciertas con seguridad (0.99 / 0.01):", round(entropia_cruzada(np.array([1, 0]), np.array([0.99, 0.01])), 3))
print("Si fallas con seguridad (0.01 / 0.99):", round(entropia_cruzada(np.array([1, 0]), np.array([0.01, 0.99])), 3))

**Ejercicio 3 · 🏫 en clase.** Escribe `paso_gradiente(parametros, gradientes, tasa)`: recibe dos diccionarios con
las mismas claves (`"W1"`, `"b1"`, `"W2"`, `"b2"`) y devuelve un diccionario **nuevo** con cada
parámetro actualizado: `nuevo = viejo - tasa * gradiente`. `tasa` es la **tasa de aprendizaje**
(el tamaño del pasito).

In [ ]:
# TODO: Recorre las claves y resta tasa * gradiente a cada parámetro
...

_p = {"W1": np.array([1.0, 2.0]), "b1": np.array([0.5])}
_g = {"W1": np.array([0.5, -1.0]), "b1": np.array([1.0])}
_nuevo = paso_gradiente(_p, _g, 0.1)
assert np.allclose(_nuevo["W1"], [0.95, 2.1]) and np.allclose(_nuevo["b1"], [0.4]), "Cada parámetro debe restar tasa * gradiente"
assert _p["W1"][0] == 1.0, "No modifiques el diccionario original: devuelve uno nuevo"

---

## 🏁 Fin de la parte de clase

- Lo que queda (los ejercicios 🏠 y el mini-reto) está en el **[cuaderno completo](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S10/2_Practica/para_casa/S10_01_red_desde_cero.ipynb)**.
- Ahí también tienes, en cada ejercicio de casa, su celda plegada «🆘 Solución de referencia» por si te quedas atascado.
